In [ ]:
! python --version

In [1]:
from __future__ import annotations

"""Utility helpers for the recipe chatbot backend.

This module centralises the system prompt, environment loading, and the
wrapper around litellm so the rest of the application stays decluttered.
"""

import os
from typing import Final, List, Dict

import litellm  # type: ignore
from dotenv import load_dotenv

# Ensure the .env file is loaded as early as possible.
load_dotenv(override=False)

# --- Constants -------------------------------------------------------------------

meal_type_options = ['entrée', 'dessert', 'main', 'beverage']
dietary_preference_options = ['vegan', 'vegetarian', 'omnivore', 'meat', 'pescatarian', 'nut-allergic']
difficulty_options = ['easy', 'medium', 'hard', 'very_hard']
mealtime_options = ["breakfast", "lunch", "dinner", "snack"]
time_required_options = ['under_30_minutes', '30_to_60_minutes', 'over_1_hour']

SYSTEM_PROMPT: Final[str] = f'''\
Given the following key Japanese recipe dimensions:
- Difficulty: one of {difficulty_options}
- Dietary Preference: one of {dietary_preference_options}
- Meal Type: one of {meal_type_options}
- Mealtime: one of {mealtime_options}
- Time Required: one of {time_required_options}

Can you provide a list of exactly 20 combinations of these dimensions? They have to make sense together.

For each combination, also provide a brief persona description that would fit that combination.

For example, here are a few combinations based on a particular persona:

1) Persona: "Beginner cook looking for quick and easy meals"
   - Difficulty: easy
   - Dietary Preference: omnivore
   - Meal Type: main
   - Mealtime: dinner
   - Time Required: under_30_minutes

2) Persona: "Health-conscious individual seeking vegetarian options"
   - Difficulty: medium
   - Dietary Preference: vegetarian
   - Meal Type: entrée
   - Mealtime: lunch
   - Time Required: 30_to_60_minutes

3) Persona: "Gourmet chef interested in complex recipes"
    - Difficulty: very_hard
    - Dietary Preference: omnivore
    - Meal Type: main
    - Mealtime: dinner
    - Time Required: over_1_hour

Remember, that we only need 20 combinations.
'''

# Fetch configuration *after* we loaded the .env file.
MODEL_NAME: Final[str] = os.environ.get("MODEL_NAME", "gpt-4o-mini")

def get_dimension_combinations() -> list[tuple[str, tuple[str, str, str, str, str]]]:
    """
    Use the SYSTEM_PROMPT and an LLM call to return a realistic list of 20 unique combinations of personas, and key Japanese
    recipe dimensions (difficulty, dietary_preference, meal_type, mealtime, time_required) as tuples.
    
    Returns:
        List of tuples where each tuple is (persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Please provide the list as a Python list of tuples, where each tuple contains:\n(persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))\n\nFor example:\n[('Beginner cook looking for quick meals', ('easy', 'omnivore', 'main', 'dinner', 'under_30_minutes')), ...]"}
    ]
    completion = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
    )
    import ast
    import re
    # Extract the list of tuples from the assistant's reply
    reply = completion["choices"][0]["message"]["content"].strip()
    # Try to extract the first Python list of tuples from the reply
    match = re.search(r'\[.*\]', reply, re.DOTALL)
    if match:
        list_str = match.group(0)
        try:
            result = ast.literal_eval(list_str)
            if isinstance(result, list) and all(isinstance(t, tuple) and len(t) == 2 for t in result):
                # Validate structure: (persona_string, (5-tuple of dimensions))
                valid = all(
                    isinstance(t[0], str) and 
                    isinstance(t[1], tuple) and 
                    len(t[1]) == 5 
                    for t in result
                )
                if valid:
                    return result
        except Exception as e:
            print(f"Error parsing result: {e}")
            pass
    # Fallback: return the raw reply if parsing fails
    return reply


In [2]:
dimension_combinations = get_dimension_combinations()
length = len(dimension_combinations) if isinstance(dimension_combinations, list) else 'N/A'
print(f"Number of combinations: {length}")

Number of combinations: 20


In [3]:
dimension_combinations

[('Beginner cook looking for quick and easy meals',
  ('easy', 'omnivore', 'main', 'dinner', 'under_30_minutes')),
 ('Health-conscious individual seeking vegetarian lunch options',
  ('medium', 'vegetarian', 'entrée', 'lunch', '30_to_60_minutes')),
 ('Gourmet chef interested in complex omnivore dinners',
  ('very_hard', 'omnivore', 'main', 'dinner', 'over_1_hour')),
 ('Vegan looking for quick breakfast ideas',
  ('easy', 'vegan', 'entrée', 'breakfast', 'under_30_minutes')),
 ('Pescatarian wanting a seafood snack with moderate effort',
  ('medium', 'pescatarian', 'snack', 'snack', '30_to_60_minutes')),
 ('Nut-allergic person seeking easy dessert recipes',
  ('easy', 'nut-allergic', 'dessert', 'snack', 'under_30_minutes')),
 ('Experienced cook preparing a vegetarian main for dinner',
  ('hard', 'vegetarian', 'main', 'dinner', 'over_1_hour')),
 ('Casual cook seeking omnivore lunch options under an hour',
  ('medium', 'omnivore', 'main', 'lunch', '30_to_60_minutes')),
 ('Busy professional 

In [4]:
import asyncio

async def generate_single_query_async(persona: str, dimensions: tuple[str, str, str, str, str]) -> str:
    """Generate a single natural language query for one persona-dimension combination asynchronously."""
    difficulty, dietary_preference, meal_type, mealtime, time_required = dimensions
    
    prompt = f'''You are helping generate a realistic user query for a recipe chatbot.

Given this user persona: "{persona}"
And these recipe requirements:
- Difficulty level: {difficulty}
- Dietary preference: {dietary_preference}
- Meal type: {meal_type}
- Mealtime: {mealtime}
- Time required: {time_required}

Write a single, natural user query that someone with this persona might ask a recipe chatbot. The query should reflect their cooking level, dietary needs, and time constraints,
but should sound natural and conversational (not mentioning the specific dimension names). Natural language have a lot of imperfections, so feel free to include small typos or
colloquial phrases. Mis-spellings and informal language are welcome. Mixed capitalisations are also common in natural language queries. You don't have to allways start the
question with `Hey`. Also, some queries are quite terse and to the point, while others may be more elaborate.

Return only the query text, no additional formatting or explanation.'''
    
    messages = [
        {"role": "system", "content": "You are an expert at generating realistic, conversational user queries for a recipe chatbot."},
        {"role": "user", "content": prompt}
    ]
    
    try:
        completion = await asyncio.get_event_loop().run_in_executor(
            None, 
            lambda: litellm.completion(model=MODEL_NAME, messages=messages)
        )
        return completion["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error generating query for {persona[:30]}...: {e}")
        return f"Error generating query for {persona}"

async def generate_natural_language_queries_async(combinations: list[tuple[str, tuple[str, str, str, str, str]]], n: int = 10) -> list[str]:
    """Use async to generate realistic natural language user queries for each combination in parallel."""
    selected_combinations = combinations[:n]
    
    print(f"Generating {len(selected_combinations)} queries asynchronously...")
    
    tasks = [
        generate_single_query_async(persona, dimensions) 
        for persona, dimensions in selected_combinations
    ]
    
    queries = await asyncio.gather(*tasks)
    return queries

In [6]:
# In Jupyter, you can await async functions directly in cells
queries = await generate_natural_language_queries_async(dimension_combinations, n=20)
print(f"\nGenerated {len(queries)} user queries:")
print("="*50)
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")
    print()

Generating 20 queries asynchronously...

Generated 20 user queries:
1. I’m kinda new to cooking, can u suggest some easy dinner ideas with meat or veggies that I can make in less than 30 mins?

2. Can you suggest a tasty vegetarian lunch recipe that's not super easy but not too hard, and takes about 30 to 60 minutes to make? Looking for something healthy and filling!

3. I’m looking for a really challenging dinner recipe that features both meat and seafood, something intricate and impressive that’ll take more than an hour to pull off. Got any ideas?

4. Got any super easy vegan breakfast ideas I can whip up in like 20 minutes or less?

5. Got any good seafood snack ideas that aren’t too easy but not crazy hard? I’ve got about half an hour to an hour to whip something up. I'm pescatarian btw!

6. Can u suggest some quick dessert snacks without nuts? I'm not great at cooking and want something easy and ready fast!

7. I’m looking for a really sturdy vegetarian main to make for dinner ton

In [7]:
# Export queries to CSV file
import csv
import os

# Create the data directory if it doesn't exist
os.makedirs('../../data', exist_ok=True)

# Write queries to CSV with the specified schema
csv_path = '../../data/sample_queries_hw2.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['id', 'query'])
    
    # Write queries with sequential IDs
    for i, query in enumerate(queries, 1):
        writer.writerow([i, query])

print(f"Exported {len(queries)} queries to {csv_path}")

Exported 20 queries to ../../data/sample_queries_hw2.csv
